# DRAFT only showing ideas

# Adaptations

In this lab we explore the main techniques to adapt foundation models to specific tasks and domains, without (or with minimal) retraining.

# 1. Prompt Engineering

Prompt engineering is the practice of designing the input text to steer a model's behavior without changing its weights. The quality and structure of the prompt has a direct impact on the output.

## 1.1 Zero-shot prompting

The model receives only the task description, with no examples.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, device_map="auto", torch_dtype="auto", trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
messages = [
    {"role": "system", "content": "You are a sentiment analysis assistant. Classify the sentiment as positive, negative or neutral."},
    {"role": "user", "content": "The movie was absolutely wonderful, I loved every minute of it."}
]
input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
outputs = model.generate(**input_ids, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 1.2 Few-shot prompting

We provide examples in the prompt so the model learns the expected input/output format in-context.

In [ ]:
messages = [
    {"role": "system", "content": "You are a sentiment analysis assistant. Classify the sentiment as positive, negative or neutral."},
    {"role": "user", "content": "I really enjoyed the food at this restaurant."},
    {"role": "assistant", "content": "positive"},
    {"role": "user", "content": "The service was terrible and the waiter was rude."},
    {"role": "assistant", "content": "negative"},
    {"role": "user", "content": "The package arrived on Tuesday."},
    {"role": "assistant", "content": "neutral"},
    {"role": "user", "content": "The movie was absolutely wonderful, I loved every minute of it."}
]
input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
outputs = model.generate(**input_ids, max_new_tokens=50)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 1.3 Chain-of-Thought (CoT)

We instruct the model to reason step by step before giving the final answer. This improves performance on tasks that require reasoning.

In [ ]:
messages = [
    {"role": "system", "content": "You are a math assistant. Think step by step before giving the final answer."},
    {"role": "user", "content": "If a train travels at 60 km/h for 2.5 hours, how far does it go?"}
]
input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
outputs = model.generate(**input_ids, max_new_tokens=256)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

# 2. Retrieval-Augmented Generation (RAG)

RAG augments the model's knowledge by retrieving relevant documents from an external source and injecting them into the prompt. This avoids hallucinations and allows the model to answer questions about data it was never trained on.

## 2.1 Simple RAG pipeline

In this minimal example we simulate retrieval by providing a context document directly in the system prompt.

In [ ]:
context = """
The Eiffel Tower was built in 1889 for the World's Fair in Paris.
It is 330 meters tall and was designed by Gustave Eiffel's engineering company.
It was originally intended to be dismantled after 20 years.
"""

messages = [
    {"role": "system", "content": f"Answer the user's question using ONLY the following context. If the answer is not in the context, say so.\n\nContext:\n{context}"},
    {"role": "user", "content": "When was the Eiffel Tower built and how tall is it?"}
]
input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to(model.device)
outputs = model.generate(**input_ids, max_new_tokens=128)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 2.2 RAG with embeddings

A real RAG system uses an embedding model to encode documents into vectors, stores them in a vector database, and retrieves the most relevant chunks at query time.

```
+------------+     +-----------+     +----------+     +-----------+
| Documents  | --> | Embedding | --> | Vector   | --> | Retrieval |
|            |     | Model     |     | Store    |     |           |
+------------+     +-----------+     +----------+     +-----+-----+
                                                            |
                                                            v
+------------+     +-----------+     +----------------------+
| User Query | --> | Embedding | --> | Top-k similar chunks |
+------------+     +-----------+     +----------+-----------+
                                                |
                                                v
                                     +----------+-----------+
                                     | LLM generates answer |
                                     | with retrieved context|
                                     +----------------------+
```

In [ ]:
# TODO: Implement RAG with a vector store (e.g. FAISS) and an embedding model

# 3. Quantization

Quantization reduces the precision of model weights (e.g. from 16-bit to 4-bit), significantly reducing memory usage and enabling larger models to run on consumer hardware. The trade-off is a small loss in quality.

## 3.1 Loading a quantized model with BitsAndBytes

`bitsandbytes` integrates with `transformers` to load models in 4-bit or 8-bit precision.

In [ ]:
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model_q = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True,
)

In [ ]:
# Compare memory usage
print(f"Original model memory: {model.get_memory_footprint() / 1e9:.2f} GB")
print(f"Quantized model memory: {model_q.get_memory_footprint() / 1e9:.2f} GB")

## 3.2 Using pre-quantized models (GGUF)

Many models on Hugging Face are already quantized and published in GGUF format. These can be loaded directly without needing to quantize yourself.

In [ ]:
# TODO: Load a GGUF quantized model

# 4. Fine-Tuning

Fine-tuning updates the model weights on a task-specific dataset. Full fine-tuning is expensive, so parameter-efficient methods like LoRA are preferred.

## 4.1 LoRA (Low-Rank Adaptation)

LoRA freezes the original weights and injects small trainable matrices into the attention layers. This drastically reduces the number of trainable parameters.

```
Original weight W (frozen)
        |
        +--- x ----> W·x  (original path)
        |                    
        +--- x --> A·x --> B·(A·x)  (LoRA path, A and B are small trainable matrices)
        |                    
        +--- output = W·x + B·A·x
```

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model_lora = get_peft_model(model, lora_config)
model_lora.print_trainable_parameters()

## 4.2 Training with SFTTrainer

`trl` provides `SFTTrainer` for supervised fine-tuning with chat-formatted datasets.

In [ ]:
# TODO: Prepare a dataset and fine-tune with SFTTrainer
# from trl import SFTTrainer, SFTConfig

## 4.3 QLoRA

QLoRA combines quantization (section 3) with LoRA: the base model is loaded in 4-bit, and only the LoRA adapters are trained in higher precision. This enables fine-tuning large models on a single GPU.

In [ ]:
# TODO: Combine BitsAndBytesConfig with LoRA for QLoRA fine-tuning

# 5. Knowledge Distillation

Knowledge distillation trains a smaller *student* model to mimic the outputs of a larger *teacher* model. The student learns from the teacher's soft probability distributions (logits), which contain richer information than hard labels.

```
+----------------+                  +----------------+
| Teacher Model  |  -- logits -->   |   Distillation |
| (large, frozen)|                  |      Loss      |
+----------------+                  +-------+--------+
                                            |
+----------------+                          v
| Student Model  |  -- logits -->   backpropagation
| (small, trains)|                  updates student
+----------------+
```

In [ ]:
# TODO: Implement a distillation loop between a teacher and student model

# Assignment #3

Same assignment with RAG